# Plan F Phase 2: REINFORCE で 重み NN を fine-tune (Colab T4 GPU)

対戦用 AI = TwoTurnPlanningAI 固定、 専用評価関数 NN を self-play snapshot から REINFORCE で学習。

**事前準備**:
1. ローカルで `scripts/collect_twoturn_snapshots.py` 実行 → `db/twoturn_snapshots.jsonl` 生成
2. snapshot を Drive `MyDrive/onepiece_nn/twoturn_snapshots.jsonl` にアップロード
3. base NN (= `db/weight_nn.pt`) も Drive `MyDrive/onepiece_nn/weight_nn.pt` にアップロード
4. このノートブックを開く、 ランタイム > T4 GPU 選択、 全セル実行
5. `MyDrive/onepiece_nn/weight_nn_rl.pt` を ローカル `db/weight_nn_rl.pt` に download
6. ローカルで mirror eval で効果確認

## 1. GPU 確認

In [ ]:
import torch
print('torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), 'GPU が有効化されていません'

## 2. Drive マウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/onepiece_nn'
SNAPSHOT_PATH = os.path.join(WORK_DIR, 'twoturn_snapshots.jsonl')
BASE_NN_PATH = os.path.join(WORK_DIR, 'weight_nn.pt')
OUTPUT_PT = os.path.join(WORK_DIR, 'weight_nn_rl.pt')

assert os.path.exists(SNAPSHOT_PATH), f'snapshot 不在: {SNAPSHOT_PATH}'
assert os.path.exists(BASE_NN_PATH), f'base NN 不在: {BASE_NN_PATH}'
print(f'snapshot size: {os.path.getsize(SNAPSHOT_PATH) // (1024*1024)} MB')
print(f'base NN size: {os.path.getsize(BASE_NN_PATH) // 1024} KB')

## 3. snapshot 読み込み + tensor 化

In [ ]:
import json, time
import numpy as np

STATE_DIM = 172
GAMMA = 0.95  # reward discount

t0 = time.time()
snapshots = []
with open(SNAPSHOT_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        try:
            snap = json.loads(line)
        except json.JSONDecodeError:
            continue
        if 'state_encoded' not in snap or 'reward' not in snap:
            continue
        snapshots.append(snap)
print(f'loaded {len(snapshots)} snapshots in {time.time()-t0:.1f}s')

# reward 分布
from collections import Counter
r_dist = Counter(s['reward'] for s in snapshots)
print(f'reward dist: {dict(r_dist)}')

# turn discount (= 試合終盤に近い snapshot の reward を強化)
# 各 snapshot で reward × γ^(max_turn - turn) を計算
for s in snapshots:
    delta = max(0, s.get('max_turn', s.get('turn', 0)) - s.get('turn', 0))
    s['discounted_reward'] = s['reward'] * (GAMMA ** delta)

In [ ]:
# tensor 化
device = torch.device('cuda')

X_np = np.zeros((len(snapshots), STATE_DIM), dtype=np.float32)
R_np = np.zeros((len(snapshots),), dtype=np.float32)
for i, snap in enumerate(snapshots):
    se = snap['state_encoded']
    if len(se) >= STATE_DIM:
        X_np[i, :] = se[:STATE_DIM]
    else:
        X_np[i, :len(se)] = se
    R_np[i] = snap['discounted_reward']

X = torch.from_numpy(X_np).to(device)
R = torch.from_numpy(R_np).to(device)
print(f'X shape: {X.shape}, R range: [{R.min().item():.2f}, {R.max().item():.2f}]')
print(f'R mean: {R.mean().item():.3f}, std: {R.std().item():.3f}')
del snapshots, X_np, R_np

## 4. WeightNN 定義 + base load

In [ ]:
import torch.nn as nn

N_WEIGHTS = 9
BASE_SCALES = torch.tensor([3000, 800, 1500, 1, 500, 1200, 1000, 600, 30000],
                            dtype=torch.float32, device=device)

class WeightNN(nn.Module):
    def __init__(self, input_dim=172, hidden_dim=256, dropout=0.1):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, hidden_dim // 4), nn.ReLU(),
        )
        self.weight_head = nn.Linear(hidden_dim // 4, N_WEIGHTS)
        self.softplus = nn.Softplus()
        self.register_buffer('base_scales', BASE_SCALES.cpu().clone())
    def forward(self, x):
        h = self.shared(x)
        raw = self.weight_head(h)
        return self.softplus(raw) * self.base_scales

model = WeightNN().to(device)
model.load_state_dict(torch.load(BASE_NN_PATH, map_location=device, weights_only=True))
model.train()
n_params = sum(p.numel() for p in model.parameters())
print(f'base model loaded, params: {n_params}')

## 5. REINFORCE 学習

勝った snapshot の weights を NN が出やすくする方向で update。 連続出力 NN なので、
Gaussian policy を仮定し、 log P(weights|state) × reward を maximize する。

実装簡略化: weights output に Gaussian noise を加えて確率化、 baseline-subtracted reward で update。

In [ ]:
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

EPOCHS = 30
BATCH = 512
LR = 1e-4
NOISE_STD = 0.3  # Gaussian noise std (= exploration)

# baseline = mean reward (= variance 削減のため)
baseline = R.mean()
advantage = R - baseline
print(f'baseline: {baseline.item():.3f}, advantage range: [{advantage.min().item():.2f}, {advantage.max().item():.2f}]')

train_ds = TensorDataset(X, advantage)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=0)

optimizer = optim.Adam(model.parameters(), lr=LR)

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    n_batches = 0
    for xb, advb in train_loader:
        optimizer.zero_grad()
        # NN(state) で mean weights、 Gaussian noise を sampled として確率化
        mu = model(xb)  # (B, 9) 期待 weights
        # sampled weights = mu + noise (= 探索)
        eps = torch.randn_like(mu) * NOISE_STD * mu.abs().mean(dim=1, keepdim=True)
        sampled = mu + eps
        # log P(sampled | mu, std=NOISE_STD * mu_mean) = -0.5 × ((sampled - mu) / std)^2
        std = NOISE_STD * mu.abs().mean(dim=1, keepdim=True).clamp(min=1.0)
        log_prob = -0.5 * ((sampled - mu) / std) ** 2
        log_prob = log_prob.sum(dim=1)  # (B,) 各 sample の log prob (Gaussian)
        # REINFORCE loss = -E[log_prob × advantage]
        loss = -(log_prob * advb).mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    avg = epoch_loss / n_batches
    print(f'  epoch {epoch+1}/{EPOCHS}: avg_loss={avg:.4f}')

torch.save(model.state_dict(), OUTPUT_PT)
print(f'\nsaved: {OUTPUT_PT}')
print('→ ローカルに db/weight_nn_rl.pt として download してください')